# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames the capstone before any modeling. Work the sections **in order**. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `framing-ml-problems` + `flyrank/flyrank-data`.

## 0. Setup (Google Colab)

Run this cell first. On Colab it clones **this** GitHub repo and installs requirements so the starter CSV is available.

In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/haideriqbal499/Flyrank_Ml_internship"
REPO_DIR = "Flyrank_Ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — clone/setup failed"
print("Starter data found. Ready for framing.")

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring**

I am starting with Lane 2 because the starter data already shows a large, messy queue of pages that *look* important but are losing visibility. An editor cannot manually scan 30,000 rows each week, and a single rule (for example, "flag everything declining") would bury high-impression page-one URLs next to low-demand noise. This lane turns scattered signals — impressions, position tier, trend, freshness, engagement — into a ranked review list with reason codes, so the first human action is obvious: refresh, expand, protect, prune, or monitor.

In [ ]:
LANE = "Lane 2: Refresh / Content Opportunity Scoring"
TASK_TYPE = "Ranking / scoring"
UNIT = "one content item (page)"
print(LANE)
print(f"Task type: {TASK_TYPE}")
print(f"Unit of analysis: {UNIT}")

## 2. The question: decision, action, cost of a wrong call

**Search question:** Which pages should an SEO/content editor review first for refresh, expansion, protection, pruning, or monitoring?

| Piece | My answer |
|---|---|
| **Unit of analysis** | One content item (one row in the starter CSV; later one `content_id` in the warehouse). |
| **Output** | A ranked action queue: priority score, suggested action, and short reason codes per page. |
| **Decision it improves** | "What do we work on this week?" when the inventory is too large to eyeball. |
| **Who acts** | A content or SEO editor (or account lead) who chooses the next batch of pages to update. |
| **Action they take** | Open the top-ranked pages, confirm the reason code, then refresh copy, expand coverage, protect a strong URL, prune/merge weak ones, or add to a watch list. |
| **Cost of a wrong call** | **False positive:** editor hours spent on a page that did not need work. **False negative:** a high-visibility page keeps sliding while the team fixes low-impact URLs. The second error is usually worse when impressions are high. |
| **Why data/ML helps** | The pattern is real but tangled: decline, demand, position, age, and engagement interact. A transparent baseline plus a simple model can combine these signals and surface *why* a page ranked high — not just a black-box score. This is decision support, not "train a model for its own sake." |

In [ ]:
frame = {
    "decision": "Which pages should an editor review first this week?",
    "actor": "SEO/content editor",
    "action": "Refresh, expand, protect, prune, or monitor specific URLs",
    "cost_if_wrong": "Wasted editor time (false positive) or missed decline on a high-visibility page (false negative)",
    "why_not_a_plain_rule": "Decline, demand, position, age, and engagement interact — no single if-statement sorts the full inventory fairly",
}
for key, value in frame.items():
    print(f"{key}: {value}")

## 3. Quick look at the data (2-3 real numbers)

Load the starter CSV and check whether Lane 2 looks worth the next 7 weeks. Expected signals (observed in this snapshot):

1. Over half of pages trend **down** on impressions.
2. Even among **high-demand** pages (≥500 impressions / 90d), most still trend down — so the priority pool is large, not a niche edge case.
3. Page-one URLs also decline often — visibility today does not mean "safe to ignore."

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_rows = len(df)
n_clients = df["client_id"].nunique()
declining_share = (df["trend_direction"] == "down").mean()

visible = df["impressions_90d"] >= 500
n_visible = int(visible.sum())
declining_visible_share = (df.loc[visible, "trend_direction"] == "down").mean()

page_one = df["position_tier"] == "page_1"
declining_page_one_share = (df.loc[page_one, "trend_direction"] == "down").mean()

print(f"Starter rows: {n_rows:,}")
print(f"Pseudonymized clients: {n_clients}")
print()
print("Supporting numbers for Lane 2:")
print(f"1) Share trending down (all pages): {declining_share:.1%}")
print(f"2) Share trending down among pages with >=500 impressions (90d): {declining_visible_share:.1%}  (n={n_visible:,})")
print(f"3) Share trending down among page-1 position tier: {declining_page_one_share:.1%}")
print()
print("Why this is not just 'train a model': these rates show a triage problem — too many candidates for hand review.")

## 4. Careful words: what I can and can't claim

**What I can say (observed / directional / decision-support):**
- In this snapshot, a large share of pages show a downward impression trend between the prior and last 30-day windows.
- Among pages with meaningful search demand (≥500 impressions in 90 days), most are still trending down — a plausible priority pool for human review.
- A ranked queue may help an editor spend limited time on higher-visibility URLs first; I will check that later with precision@K and a hand review of the top 20.

**What I will not claim:**
- That a refresh will recover traffic (no causal proof from observational data).
- That I am predicting Google's algorithm or ranking system.
- That `trend_direction` or product flags like `health_score` are unbiased ground truth — eventual labels must come from observed future windows, not copied rules.
- That results from 32 anonymized clients generalize to every site without validation.

In [ ]:
# Gotcha check from flyrank-data skill: avg_position = 0 means "no data", not rank zero
zero_position_rows = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position = 0 (missing position data): {zero_position_rows:,}")
print("Reminder: trend_direction / trend_pct are snapshot labels — never use them as model features.")
print("Frame status: provisional Lane 2 — may confirm or change through Week 4.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors in **Colab** (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Saved from Colab to GitHub under `work/notebooks/` — then submit your repo URL on the card. Done.

**Colab save steps:** File → Save a copy in GitHub → authorize the same GitHub account → path `work/notebooks/w01_research_question.ipynb` → branch `main` → OK. Open the file on GitHub and confirm **cell outputs are visible**.